In [1]:
# Push tiny ARMT (NeoX) as a single-file modeling module to avoid external imports
import os, json, shutil
from pathlib import Path
from tempfile import TemporaryDirectory

import torch
from huggingface_hub import HfApi, upload_folder
from transformers import AutoConfig, AutoModelForCausalLM

# 1) Build a tiny GPT-NeoX backbone config
base_cfg = AutoConfig.for_model(
    "gpt_neox",
    hidden_size=128,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=512,
    max_position_embeddings=512,
)

# 2) Build ARMT model from local code
from modeling_amt.model import ARMTForCausalLM, ARMTConfig
armt_cfg = ARMTConfig(
    base_model_config=base_cfg,
    num_mem_tokens=16,
    d_mem=32,
    segment_size=128,
    segment_alignment="left",
    sliding_window=False,
    layers_attr="gpt_neox.layers",
    wrap_pos=False,
    correction=True,
    n_heads=1,
    use_denom=True,
    gating=False,
    freeze_mem=False,
    act_on=True,
    max_hop=4,
    act_type="layer",
    act_format="linear",
    noisy_halting=False,
    constant_depth=False,
    time_penalty=0.0,
)
model = ARMTForCausalLM(armt_cfg)
model.eval()

# 3) Package as single-file modeling module for Hub (no external modeling_amt import)
repo_id = "irodkin/armt-neox-tiny-singlefile"
local_modeling_dir = Path("./modeling_amt")

with TemporaryDirectory() as tmpdir:
    repo_dir = Path(tmpdir) / "repo"
    repo_dir.mkdir(parents=True, exist_ok=True)

    # Save model weights and config
    model.save_pretrained(repo_dir)

    # Inline ARMT code into one file
    act_utils_path = local_modeling_dir / "act_utils.py"
    lm_path       = local_modeling_dir / "language_modeling.py"
    model_path    = local_modeling_dir / "model.py"

    act_code = act_utils_path.read_text()
    lm_code  = lm_path.read_text()
    model_code = model_path.read_text()

    # Make inlined code self-contained
    lm_code = lm_code.replace("from modeling_amt.act_utils import", "# inlined act_utils: removed import")
    model_code = model_code.replace("from modeling_amt.language_modeling import", "# inlined language_modeling: removed import")

    single_file = repo_dir / "modeling_armt.py"
    single_file.write_text(
        "# === Inlined ARMT for HF Hub (single-file) ===\n\n"
        + "# ---- act_utils.py ----\n" + act_code + "\n\n"
        + "# ---- language_modeling.py ----\n" + lm_code + "\n\n"
        + "# ---- model.py ----\n" + model_code + "\n"
    )

    # Update auto_map to point to single file
    cfg_path = repo_dir / "config.json"
    cfg = json.loads(cfg_path.read_text())
    cfg["architectures"] = ["ARMTForCausalLM"]
    cfg["auto_map"] = {
        "AutoConfig": "modeling_armt.ARMTConfig",
        "AutoModelForCausalLM": "modeling_armt.ARMTForCausalLM",
    }
    cfg_path.write_text(json.dumps(cfg, indent=2))

    # README
    (repo_dir / "README.md").write_text(
        f"# {repo_id}\n\nTiny GPT-NeoX + ARMT single-file model. Load with trust_remote_code=True."
    )

    # 4) Push to Hub
    api = HfApi()
    api.create_repo(repo_id, private=True, exist_ok=True)
    upload_folder(
        folder_path=str(repo_dir),
        repo_id=repo_id,
        repo_type="model",
        commit_message="Upload single-file ARMT (GPT-NeoX tiny)",
    )

print(f"Pushed to {repo_id}")

*** Setting default RWKV_MY_TESTING = x060 ***
[2025-08-11 18:22:30,628] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/ivan.rodkin/miniconda3/envs/env/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
nvcc warning : incompatible redefinition for option 'compiler-bindir', the last value of this option was used
/home/ivan.rodkin/miniconda3/envs/env/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: /home/ivan.rodkin/miniconda3/envs/env/lib/libcufile.so: undefined reference to `dlvsym'
/home/ivan.rodkin/miniconda3/envs/env/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: /home/ivan.rodkin/miniconda3/envs/env/lib/libcufile.so: undefined reference to `dlopen'
/home/ivan.rodkin/miniconda3/envs/env/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: /home/ivan.rodkin/miniconda3/envs/env/lib/libcufile.so: undefined reference to `dlclose'
/home/ivan.rodkin/miniconda3/envs

[RWKV.model] Running RWKV infctx using 'torch-jit' with torch '2.3.1+cu121'
[RWKV.model] Running RWKV infctx using 'torch-jit' with torch '2.3.1+cu121'
*** Can't import RWKV model ***


/home/ivan.rodkin/miniconda3/envs/env/lib/python3.9/site-packages/huggingface_hub/hf_api.py:9696: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


model.safetensors:   0%|          | 0.00/55.2M [00:00<?, ?B/s]

Pushed to irodkin/armt-neox-tiny-singlefile


In [2]:
# 5) Load back (remote code uses the single-file module)
loaded = AutoModelForCausalLM.from_pretrained(repo_id, trust_remote_code=True)
print(type(loaded))

modeling_armt.py:   0%|          | 0.00/73.3k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/irodkin/armt-neox-tiny-singlefile:
- modeling_armt.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


*** Can't import RWKV model ***


model.safetensors:   0%|          | 0.00/55.2M [00:00<?, ?B/s]

<class 'transformers_modules.irodkin.armt-neox-tiny-singlefile.9a1ef74d6afec0503a0b9bd86d8c85f21a985527.modeling_armt.ARMTForCausalLM'>


In [5]:
loaded

ARMTForCausalLM(
  (armt): AssociativeRecurrentWrapper(
    (memory_cell): AssociativeMemoryCell(
      (model): GPTNeoXForCausalLM(
        (gpt_neox): GPTNeoXModel(
          (embed_in): Embedding(50432, 128)
          (emb_dropout): Dropout(p=0.0, inplace=False)
          (layers): ModuleList(
            (0-3): 4 x AdaptiveAssociativeLayerWrapper2(
              (W_mq): Linear(in_features=128, out_features=32, bias=False)
              (W_mk): Linear(in_features=128, out_features=32, bias=False)
              (W_mv): Linear(in_features=128, out_features=128, bias=False)
              (W_mb): Linear(in_features=128, out_features=1, bias=True)
              (layer): GPTNeoXLayer(
                (input_layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
                (post_attention_layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
                (post_attention_dropout): Dropout(p=0.0, inplace=False)
                (post_mlp_dropout): Dropout(p=0.0,